# Scrap notebooks

This notebook is just a compilation of scrapy tests to view functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sympy as sym

In [ ]:
%matplotlib widget

In [ ]:
phi = 0.3
phi_min = 0.1
phi_max = 0.9

phi_var = sym.symbols('varphi', positive=True) # phi is lower than 1 but no assumption system

alpha = 2
beta = 1
gamma = 4
delta = beta * gamma / (beta + gamma)

alpha_var, beta_var, gamma_var = sym.symbols('alpha beta gamma', positive=True)
delta_expr = beta_var * gamma_var / (beta_var + gamma_var)
delta_var = sym.symbols('delta', positive=True) # delta_var and delta_expr cannot be linked

f = 0.5
f_min = 0
f_max = 10

f_var = sym.symbols('f', nonnegative=True)

E = 0.3
E_min = 0
E_max = 10

E_var = sym.symbols('E', nonnegative=True)

N = 10000
N_min = 1
N_max = 100000

N_var = sym.symbols('N', positive=True)

s = 5000
s_min = 2000
s_max = 10000

s_var = sym.symbols('s', positive=True)

M = 3
M_min = 1
M_max = 8

M_var = sym.symbols('M', positive=True)

Delta_d = 1
#Delta_d = -2 * delta * N / (phi * s)
Delta_d_min = -5
Delta_d_max = 15

Delta_d_var = sym.symbols('Delta_d', real=True)

Delta_p = 3.2
#Delta_p = Delta_d + f + E + delta * N / (phi * s) # to delete
Delta_p_min = -5
Delta_p_max = 20

Delta_p_var = sym.symbols('Delta_p', real=True)

In [ ]:
dim = 200

# first depending variables
N_ds = np.linspace(0, N, dim)
N_ps = np.linspace(0, N, dim)

N_d_var = sym.symbols('N_d', nonnegative=True)
N_p_var = sym.symbols('N_p', nonnegative=True)
N_s_expr = N_var - N_d_var - N_p_var
N_s_var = sym.symbols('N_s', nonnegative=True) # N_s_var and N_s_expr cannot be linked

N_ds, N_ps = np.meshgrid(N_ds, N_ps)
N_ss = N - N_ds - N_ps

mask = (N_ds >= 0) & (N_ps >= 0) & (N_ss >= 0) & (N_ps <= M * N_ds)

surface = np.zeros((dim, dim), dtype="float")

surface[~mask] = np.nan

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(N_ds, N_ps, surface, cmap=plt.cm.viridis, linewidth=0.2)

ax.set_xlabel('N_d')
ax.set_ylabel('N_p')


In [ ]:
SC = (delta / s) * ((N_ss)**2/(1-phi) + (N_ds**2 + N_ds * N_ps)/phi)\
    + (N_ss + N_ds) * (f + E) + N_ds * Delta_d + N_ps * Delta_p

SC_expr = (delta_var / s_var) * ((N_s_expr)**2/(1-phi_var) + (N_d_var**2 + N_d_var * N_p_var)/phi_var)\
    + (N_s_expr + N_d_var) * (f_var + E_var) + N_d_var * Delta_d_var + N_p_var * Delta_p_var

In [ ]:
SC_expr

In [ ]:
fig = plt.figure()
# some line of codes to see the saddle point

# N_ds = np.linspace(1821, 1822, dim)
# N_ps = np.linspace(803, 804, dim)
# N_ds, N_ps = np.meshgrid(N_ds, N_ps)
# N_ss = N - N_ds - N_ps
# SC = (delta / s) * ((N_ss)**2/(1-phi) + (N_ds**2 + N_ds * N_ps)/phi)\
#    + (N_ss + N_ds) * (f + E) + N_ds * Delta_d + N_ps * Delta_p
# 
# SC[(N_ds > 1821.3) & (N_ps > 1821.3- N_ds + 803.8)] = np.nan
# SC[(N_ds <1821.7) & (N_ps < 1821.7- N_ds + 803)] = np.nan
SC[~mask] = np.nan
ax = fig.add_subplot(projection='3d')
ax.plot_surface(N_ds, N_ps, SC, cmap=plt.cm.viridis, linewidth=0.2)
ax.set_xlabel('N_d')
ax.set_ylabel('N_p')
ax.view_init(20, 0, 0)

In [ ]:
grad_SC = sym.Matrix([SC_expr.diff(var) for var in (N_d_var, N_p_var)])
grad_SC

In [ ]:
hessian_SC = sym.Matrix([[SC_expr.diff(var1).diff(var2) for var2 in (N_d_var, N_p_var)] for var1 in (N_d_var, N_p_var)])
hessian_SC

In [ ]:
hessian_SC_eigen = hessian_SC.eigenvals()
list(hessian_SC_eigen.keys())[0] # which is negative

In [ ]:
list(hessian_SC_eigen.keys())[1] # which is positive

In [ ]:
N_ps_1dim = N_ps[:, 0]
lhs = (f + E + delta * N / s + Delta_d * (1 + phi) / 2) * np.ones(dim)
rhs = delta * N_ps_1dim / ((1-phi) * s)  * (2 - 1/2 * (1+phi)*(1+phi) / phi) + Delta_p

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 5), sharex=True, tight_layout=True)
ax1.plot(N_ps_1dim, lhs, label="lhs")
ax1.plot(N_ps_1dim, rhs, label="rhs")

N_ds_1dim_of_N_ps = - N_ps_1dim / 2 * (1+phi) + phi*(1-phi) * (N / (1-phi) - Delta_d * s / (2 * delta))
ax2.plot(N_ps_1dim, N_ds_1dim_of_N_ps, label="N_d")

ax2.set_xlabel('N_p')
ax2.set_ylabel('N_d')

In [ ]:
hessian_SC_eigenvects = hessian_SC.eigenvects()
eigen_values = [hessian_SC_eigenvects[k][0] for k in range(len(hessian_SC_eigenvects)) for l in range (hessian_SC_eigenvects[k][1])]
eigen_vects = [hessian_SC_eigenvects[k][2] for k in range(len(hessian_SC_eigenvects))]

def list_of_vector_to_matrix(list_of_vect):
    matrix = list_of_vect[0]
    for k in range(1, len(list_of_vect)):
        matrix = matrix.row_join(list_of_vect[k])

    return matrix

eigen_vects = [list_of_vector_to_matrix(vect) for vect in eigen_vects]
eigen_vects = list_of_vector_to_matrix(eigen_vects)

diag_matrix = sym.Matrix.diag(*[eigen_values])
# eigen_vects_check, diag_check = hessian_SC.diagonalize()


#### just a few commands to understand basic linear algebra

In [ ]:
# diag_matrix

In [ ]:
# eigen_vects

In [ ]:
# (eigen_vects.inv() * hessian_SC * eigen_vects).applyfunc(sym.cancel)

In [ ]:
# diag_matrix.applyfunc(sym.cancel)

In [ ]:
# eigen_vects_check, diag_check = hessian_SC.diagonalize()

In [ ]:
# (eigen_vects_check.inv() * hessian_SC * eigen_vects_check).applyfunc(sym.cancel)

Ok, j’ai tout compris

Maintenant, on va trouver les solutions.

## Solving

Writing Lagrangian

In [ ]:
# lamb = sym.symbols("lambda")
mu_s, mu_d, mu_p, mu_v = sym.symbols("mu_s mu_d mu_p mu_v", positive=True)

Lag_expr = SC_expr \
    - mu_s * N_s_expr \
    - mu_d * N_d_var \
    - mu_p * N_p_var \
    - mu_v * (M_var * N_d_var - N_p_var)
#     - lamb * (N_var - N_d_var - N_p_var - N_s_expr) \

In [ ]:
Lag_expr

In [ ]:
conditions = [Lag_expr.diff(var) for var in (N_d_var, N_p_var, mu_s, mu_d, mu_p, mu_v)]

In [ ]:
conditions[0]

In [ ]:
conditions[1]

In [ ]:
conditions[2]

In [ ]:
conditions[3]

In [ ]:
conditions[4]

In [ ]:
conditions[5]

In [ ]:
SC_phi_1_expr = (delta_var / s_var) * ((N_d_var**2 + N_d_var * (N_var - N_d_var)))\
    + (N_d_var) * (f_var + E_var) + N_d_var * Delta_d_var + (N_var - N_d_var) * Delta_p_var

In [ ]:
SC_phi_1_expr = SC_phi_1_expr.expand()

In [ ]:
SC_phi_1_expr

In [ ]:
SC_phi_1_expr_diff = SC_phi_1_expr.diff(N_d_var)
SC_phi_1_expr_diff

* Si c’est positif --> on va au min de $\Delta_d$, c’est-à-dire $N_d = \frac{N}{M+1}$.
* Si c’est négatif --> On va au max, c’est-à-dire $N_d = N$

### Maintenant on résoud pour $\varphi \in ]0, 1[$

In [ ]:
SC_expr

In [ ]:
conditions[0]

In [ ]:
conditions[1]

Many cases

| $\mu_s >0$ and $N_d + N_p = N$ | $\mu_d > 0$ and $N_d = 0$ | $\mu_p > 0$ and $N_p = 0$ | $\mu_v > 0$ and $M N_d = N_p$ | Comment |
| ------------------------------ | ------------------------- | ------------------------- | ----------------------------- | ------- |
| False | False | False | False | Saddle point so not candidate for optimum &rarr; not explored.|
| False | False | False | True | **j/** We find $N_d = \frac{N \varphi}{M \varphi + 1} + \frac{s \left(M \left(- \varphi^{2} + \varphi\right) \left(- \Delta_{p} + E + f\right) + \Delta_d \left(\varphi^{2} - \varphi\right)\right)}{2 \delta \left(M + 1\right) \left(M \varphi + 1\right)}$, $N_p = MN_d$. To which we add the conditions $0 \leq N_d \leq \frac{N}{M+1}$ and $\frac{2 N \delta \left(M + 1\right)}{s} + \Delta_d \left(2 M \varphi + \varphi + 1\right) + \left(- \Delta_{p} + E + f\right) \left(M \varphi + M + 2\right) >0 $ |
| False | False | True | False | **i/**  We find $N_p = 0$ and $N_d = \varphi N \left( 1 - \left. \Delta_d \middle/ \frac{2\delta N}{(1-\varphi)s} \right. \right)$. Conditions $0 \leq N_d \leq N$ give $1 - 1/\varphi \leq \left. \Delta_d \middle/ \frac{2\delta N}{(1-\varphi)s} \right. \leq 1$. Additionnaly, we get $2\Delta_p > 2E + 2f + \Delta_d (1+\varphi) + \frac{2\delta N}{s}$. Here intuitions are harder to grasp… |
| False | False | True | True | **h/** We find $N_s = N$. Conditions are $M\Delta_p + \Delta_d > M(E + f) + (M+1)\frac{2\delta N}{(1-\varphi) s}$ and $\Delta_d > \frac{2\delta N}{(1-\varphi)s}$. Other conditions ($N_s \geq 0$ and $N_d \geq 0$) are satisfied. |
| False | True | False | False | **g/** Helped by the two additional conditions ($M N_d \geq N_p \geq 0$), we find $N_s = N$. Conditions are $\Delta_p = E + f + \frac{2\delta N}{(1-\varphi)s}$ and $\Delta_d > \frac{2\delta N}{(1-\varphi)s}$. (This implies $M\Delta_p + \Delta_d > M(E + f) + (M+1)\frac{2\delta N}{(1-\varphi) s}$)|
| False | True | False | True | **f/** We find $N_s = N$. Conditions are $M\Delta_p + \Delta_d + (M+1)\frac{2\delta N}{(1-\varphi) s}$ and $-\Delta_p + E + f + \frac{2\delta N}{(1-\varphi)s}$. (This implies $\Delta_d > \frac{2\delta N}{(1-\varphi)s}$.) Other condition ($N_s \geq 0$ and $N_p \geq 0$) are satisfied.|
| False | True | True | False | **e/** We find $N_s = N$. Conditions are $\Delta_d > \frac{2\delta N}{(1-\varphi)s}$ and $\Delta_p - E - f - \frac{2\delta N}{(1-\varphi)s}$. (This implies $M\Delta_p + \Delta_d > M(E + f) + (M+1)\frac{2\delta N}{(1-\varphi) s}$.) Other conditions ($N_s \geq 0$ and $MN_d \geq N_p$) are satisfied.|
| False | True | True | True | **d/** we find $N_s = N$, conditions are $\Delta_d > \frac{2\delta N}{(1-\varphi)s}$ and $M\Delta_p + \Delta_d > M(E + f) + (M+1)\frac{2\delta N}{(1-\varphi) s}$. Other condition ($N_s \geq 0$) is satisfied. (See code for more details cause it is parametric initially) |
| True | False | False | False | **c/** If $f + E + \Delta_d + \frac{\delta N}{\varphi s} = \Delta_p$, all mixes where $N_p = N - N_d$ is solution. Condition is  $\Delta_d + \frac{2\delta N}{\varphi s} < \frac{\delta N_p}{\varphi s}$. As long as other conditions ($N_d \geq 0$, $N_p \geq 0$, $M N_d \geq N_p$) are satisfied. If the equality is not satisfied, there is no solution. Due to equality conditions, Sympy finds no solution. |
| True | False | False | True | **b/** $N_s = 0$, $N_d = \frac{N}{M + 1}$, $N_p = \frac{M N}{M + 1}$. Conditions are $M(f+E) > M\Delta _p + \Delta_d + \frac{2\delta N}{\varphi s}$ and $E + f + \Delta_d + \frac{\delta N}{\varphi s} > \Delta _p$. Other conditions ($N_d \geq 0$ and $N_p \geq 0$) are satisfied. |
| True | False | True | False | **a/** $N_d = N$, $N_p = N_s = 0$. Conditions are $- \frac{\delta N}{\varphi s} - \Delta_d > 0$ and $\Delta_{p} > E + \frac{\delta N}{\varphi s} + \Delta_d + f$. Other conditions ($N_d \geq 0$ and $M N_d \geq N_p$ are satisfied). |
| True | False | True | True | Impossible ($N_d = N$ and $MN_d = N_p = 0$) &rarr; not explored |
| True | True | False | False | Impossible ($N_p = N$ without drivers) &rarr; not explored |
| True | True | False | True | Impossible ($N = 0$) &rarr; not explored |
| True | True | True | False | Impossible ($N = 0$) &rarr; not explored |
| True | True | True | True | Impossible ($N = 0$) &rarr; not explored |

In [ ]:
condition_N_d = conditions[0]
condition_N_p = conditions[1]
condition_mu_s = conditions[2]
condition_mu_d = conditions[3]
condition_mu_p = conditions[4]
condition_mu_v = conditions[5]

### Case a

In [ ]:
condition0 = condition_N_d.subs({mu_d: 0, mu_v: 0})
condition1 = condition_N_p.subs({mu_d: 0, mu_v: 0})
condition2 = condition_mu_s.subs({mu_d: 0, mu_v: 0})
condition3 = condition_mu_p.subs({mu_d: 0, mu_v: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
condition3

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2, condition3], (N_d_var, N_p_var, mu_s, mu_p))
sols

In [ ]:
mu_s_sol = sols.args[0].args[2]
mu_p_sol = sols.args[0].args[3]
mu_s_sol = mu_s_sol.expand()
print(sym.latex(mu_s_sol))
mu_s_sol

In [ ]:
mu_p_sol = mu_p_sol.expand()
print(sym.latex(mu_p_sol))
mu_p_sol

### Case b

In [ ]:
condition0 = condition_N_d.subs({mu_d: 0, mu_p: 0})
condition1 = condition_N_p.subs({mu_d: 0, mu_p: 0})
condition2 = condition_mu_s.subs({mu_d: 0, mu_p: 0})
condition3 = condition_mu_v.subs({mu_d: 0, mu_p: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
condition3

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2, condition3], (N_d_var, N_p_var, mu_s, mu_v))
print(sym.latex(sols.args[0].args[0]))
print(sym.latex(sols.args[0].args[1]))
sols

In [ ]:
mu_s_sol = sols.args[0].args[2]
mu_v_sol = sols.args[0].args[3]
mu_s_sol = mu_s_sol.simplify()
print(sym.latex(mu_s_sol))
mu_s_sol

In [ ]:
mu_v_sol = mu_v_sol.simplify()
print(sym.latex(mu_v_sol))
mu_v_sol

### Case c

In [ ]:
condition0 = condition_N_d.subs({mu_d: 0, mu_p: 0, mu_v: 0})
condition1 = condition_N_p.subs({mu_d: 0, mu_p: 0, mu_v: 0})
condition2 = condition_mu_s.subs({mu_d: 0, mu_p: 0, mu_v: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2], (N_d_var, N_p_var, mu_s))
#print(sym.latex(sols.args[0].args[0]))
#print(sym.latex(sols.args[0].args[1]))
sols

### Case d

In [ ]:
condition0 = condition_N_d.subs({mu_s: 0})
condition1 = condition_N_p.subs({mu_s: 0})
condition2 = condition_mu_d.subs({mu_s: 0})
condition3 = condition_mu_p.subs({mu_s: 0})
condition4 = condition_mu_v.subs({mu_s: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
condition3

In [ ]:
condition4

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2, condition3, condition4], (N_d_var, N_p_var, mu_d, mu_p, mu_v))
#print(sym.latex(sols.args[0].args[0]))
#print(sym.latex(sols.args[0].args[1]))
sols

### Case e

In [ ]:
condition0 = condition_N_d.subs({mu_s: 0, mu_v: 0})
condition1 = condition_N_p.subs({mu_s: 0, mu_v: 0})
condition2 = condition_mu_d.subs({mu_s: 0, mu_v: 0})
condition3 = condition_mu_p.subs({mu_s: 0, mu_v: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
condition3

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2, condition3], (N_d_var, N_p_var, mu_d, mu_p))
#print(sym.latex(sols.args[0].args[0]))
#print(sym.latex(sols.args[0].args[1]))
sols

### Case f

In [ ]:
condition0 = condition_N_d.subs({mu_s: 0, mu_p: 0})
condition1 = condition_N_p.subs({mu_s: 0, mu_p: 0})
condition2 = condition_mu_d.subs({mu_s: 0, mu_p: 0})
condition3 = condition_mu_v.subs({mu_s: 0, mu_p: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
condition3

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2, condition3], (N_d_var, N_p_var, mu_d, mu_v))
#print(sym.latex(sols.args[0].args[0]))
#print(sym.latex(sols.args[0].args[1]))
sols

### Case g

In [ ]:
condition0 = condition_N_d.subs({mu_s: 0, mu_p: 0, mu_v: 0})
condition1 = condition_N_p.subs({mu_s: 0, mu_p: 0, mu_v: 0})
condition2 = condition_mu_d.subs({mu_s: 0, mu_p: 0, mu_v: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2], (N_d_var, N_p_var, mu_d))
#print(sym.latex(sols.args[0].args[0]))
#print(sym.latex(sols.args[0].args[1]))
sols

In [ ]:
condition0.subs({N_d_var: sols.args[0].args[0], N_p_var: sols.args[0].args[1]})\
    .subs({Delta_p_var: E_var + f_var + 2 * delta_var * N_var / ((1-phi_var) * s_var)}).simplify()

In [ ]:
condition0.subs({N_d_var: sols.args[0].args[0], N_p_var: sols.args[0].args[1]})\
    .subs({Delta_p_var: E_var + f_var + 2 * delta_var * N_var / ((1-phi_var) * s_var)}).simplify()

In [ ]:
condition_N_d

In [ ]:
condition_N_p

### Case h

In [ ]:
condition0 = condition_N_d.subs({mu_s: 0, mu_d: 0})
condition1 = condition_N_p.subs({mu_s: 0, mu_d: 0})
condition2 = condition_mu_p.subs({mu_s: 0, mu_d: 0})
condition3 = condition_mu_v.subs({mu_s: 0, mu_d: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
condition3

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2, condition3], (N_d_var, N_p_var, mu_p, mu_v))
#print(sym.latex(sols.args[0].args[0]))
#print(sym.latex(sols.args[0].args[1]))
sols

### Case i

In [ ]:
condition0 = condition_N_d.subs({mu_s: 0, mu_d: 0, mu_v: 0})
condition1 = condition_N_p.subs({mu_s: 0, mu_d: 0, mu_v: 0})
condition2 = condition_mu_p.subs({mu_s: 0, mu_d: 0, mu_v: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2], (N_d_var, N_p_var, mu_p))
#print(sym.latex(sols.args[0].args[0]))
#print(sym.latex(sols.args[0].args[1]))
sols_i = sols
sols

In [ ]:
(phi_var * N_var * (1 - Delta_d_var / (2 * delta_var * N_var / ((1 - phi_var) * s_var)))).expand() - sols.args[0].args[0].expand()

### Case j

In [ ]:
condition0 = condition_N_d.subs({mu_s: 0, mu_d: 0, mu_p: 0})
condition1 = condition_N_p.subs({mu_s: 0, mu_d: 0, mu_p: 0})
condition2 = condition_mu_v.subs({mu_s: 0, mu_d: 0, mu_p: 0})

In [ ]:
condition0

In [ ]:
condition1

In [ ]:
condition2

In [ ]:
sols = sym.linsolve([condition0, condition1, condition2], (N_d_var, N_p_var, mu_v))
#print(sym.latex(sols.args[0].args[0]))
#print(sym.latex(sols.args[0].args[1]))
sols_j = sols
sols

In [ ]:
num = sols.args[0].args[0].args[1]\
    .collect(Delta_p_var)\
    .collect(E_var)\
    .collect(f_var)\
    .collect(Delta_d_var)\
    .collect(2*delta_var * N_var * phi_var)\
    .collect(s_var)\
    .collect(-M_var * phi_var * phi_var + M_var * phi_var)\
    .collect(M_var)

num

In [ ]:
dem = sols.args[0].args[0].args[0].args[0]\
    .factor()

dem

In [ ]:
print(sym.latex(num.args[0] / dem + num.args[1] / dem))
num.args[0] / dem + num.args[1] / dem

In [ ]:
num.args[1] / dem

In [ ]:
num = sols.args[0].args[2].args[1]\
    .collect(E_var)\
    .collect(f_var)\
    .collect(Delta_d_var)\
    .collect(Delta_p_var)\
    .collect(M_var * s_var * phi_var + M_var * s_var + 2 * s_var)\
    .collect(2 * N_var * delta_var)\
    .collect(s_var)
    

num

In [ ]:
dem = sols.args[0].args[2].args[0].args[0]\
    .factor()

dem

In [ ]:
print(sym.latex(num.args[0] / s_var + num.args[1] / s_var))

num.args[0] / s_var + num.args[1] / s_var

### Plotting solutions

In [ ]:
SC_eval = sym.lambdify(
    (N_d_var, N_p_var, delta_var, s_var, M_var, f_var, E_var, phi_var, N_var, Delta_d_var, Delta_p_var),
    SC_expr,
    modules='numpy')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.set_xlabel(r'$\Delta_d$')
ax.set_ylabel(r'$\Delta_p$')

N_value = 10000
s_value = 2000
alpha_value = 2
beta_value = 1
gamma_value = 4
delta_value = beta_value * gamma_value / (beta_value + gamma_value)
phi_value = 0.3
f_value = 1
E_value = 1
M_value = 3

#eps = 0.1
eps = 0.5
dim = 2000
alpha = 0.4

#Delta_d_value = np.linspace(-16.75, -16.25, dim)
#Delta_p_value = np.linspace(-1.3, -1.5, dim)

Delta_d_value = np.linspace(-50, 50, dim)
Delta_p_value = np.linspace(-50, 50, dim)

Delta_d_value, Delta_p_value = np.meshgrid(Delta_d_value, Delta_p_value)

N_d_a_value = np.full((dim, dim), N_value)
N_p_a_value = np.full((dim, dim), 0)
SC_a_value = SC_eval(
    N_d_a_value,
    N_p_a_value,
    delta_value,
    s_value,
    M_value,
    f_value,
    E_value,
    phi_value,
    N_value,
    Delta_d_value,
    Delta_p_value
)
#SC_a_value = np.zeros((dim, dim))
SC_a_value[-delta_value * N_value / (phi_value * s_value) - Delta_d_value <=0] = np.nan
SC_a_value[Delta_p_value - E_value - f_value - Delta_d_value - delta_value * N_value / (phi_value * s_value) <= 0] = np.nan
ax.plot_surface(Delta_d_value, Delta_p_value, SC_a_value, label="a", alpha=alpha)



N_d_b_value = np.full((dim, dim), N_value / (M_value + 1))
N_p_b_value = M_value * N_d_b_value
SC_b_value = SC_eval(
    N_d_b_value,
    N_p_b_value,
    delta_value,
    s_value,
    M_value,
    f_value,
    E_value,
    phi_value,
    N_value,
    Delta_d_value,
    Delta_p_value
)
#SC_b_value = np.zeros((dim, dim))
SC_b_value[M_value * (f_value + E_value - Delta_p_value) - Delta_d_value - 2 * delta_value * N_value / (phi_value * s_value) <= 0] = np.nan
SC_b_value[E_value + f_value + Delta_d_value + delta_value * N_value / (phi_value * s_value) - Delta_p_value <= 0] = np.nan
ax.plot_surface(Delta_d_value, Delta_p_value, SC_b_value, label="b", alpha=alpha)


N_d_c_value = np.full((dim, dim), N_value / (M_value + 1))
N_p_c_value = M * N_d_c_value
#N_d_c_value = np.full((dim, dim), N_value)
#N_p_c_value = np.full((dim, dim), 0)
SC_c_value = SC_eval(
    N_d_c_value,
    N_p_c_value,
    delta_value,
    s_value,
    M_value,
    f_value,
    E_value,
    phi_value,
    N_value,
    Delta_d_value,
    Delta_p_value
)
#SC_c_value = np.zeros((dim, dim))
SC_c_value[np.abs(f_value + E_value + Delta_d_value + delta_value * N_value / (phi_value * s_value) - Delta_p_value) > eps] = np.nan
SC_c_value[Delta_d_value + delta_value * (2 * N_value - N_p_c_value) / (phi_value * s_value) >= 0 ] = np.nan
ax.plot_surface(Delta_d_value, Delta_p_value, SC_c_value, label="c", alpha=alpha)

N_d_defgh_value = np.full((dim, dim), 0)
N_p_defgh_value = np.full((dim, dim), 0)
SC_defgh_value = SC_eval(
    N_d_defgh_value,
    N_p_defgh_value,
    delta_value,
    s_value,
    M_value,
    f_value,
    E_value,
    phi_value,
    N_value,
    Delta_d_value,
    Delta_p_value
)
#SC_defgh_value = np.zeros((dim, dim))
SC_defgh_value[M_value * (Delta_p_value - f_value - E_value) + Delta_d_value - (M_value + 1) * 2 * delta_value * N_value / ((1 - phi_value) * s_value) <= 0] = np.nan
SC_defgh_value[Delta_d_value - 2 * delta_value * N_value / ((1 - phi_value) * s_value) <= 0] = np.nan

ax.plot_surface(Delta_d_value, Delta_p_value, SC_defgh_value, label="defgh", alpha=alpha)


N_d_i_value = N_value * phi_value * (1 - Delta_d_value / (2 * delta_value * N_value / ((1 - phi_value) * s_value)))
N_p_i_value = np.full((dim, dim), 0)
SC_i_value = SC_eval(
    N_d_i_value,
    N_p_i_value,
    delta_value,
    s_value,
    M_value,
    f_value,
    E_value,
    phi_value,
    N_value,
    Delta_d_value,
    Delta_p_value
)
#SC_i_value = np.zeros((dim, dim))
SC_i_value[0 > N_d_i_value] = np.nan
SC_i_value[N_value < N_d_i_value] = np.nan
SC_i_value[2 * (Delta_p_value - E_value - f_value) - Delta_d_value * (1 + phi_value) - 2 * delta_value * N_value / s_value <= 0] = np.nan
ax.plot_surface(Delta_d_value, Delta_p_value, SC_i_value, label="i", alpha=alpha)

N_d_j_value = N_value * phi_value / (M_value * phi_value + 1) +\
    s_value*(M*phi_value * (1 - phi_value) * (-Delta_p_value + E_value + f_value) - Delta_d_value * phi_value * (1 - phi_value)) /\
    (2 * delta_value * (M_value + 1)*(M_value * phi_value + 1))
N_p_j_value = M_value * N_d_j_value
SC_j_value = SC_eval(
    N_d_j_value,
    N_p_j_value,
    delta_value,
    s_value,
    M_value,
    f_value,
    E_value,
    phi_value,
    N_value,
    Delta_d_value,
    Delta_p_value
)
#SC_j_value = np.zeros((dim, dim))
SC_j_value[N_d_j_value < 0] = np.nan # should be < but this is for testing
SC_j_value[N_d_j_value > N_value / (M_value + 1)] = np.nan
SC_j_value[2 * N_value * delta_value * (M_value + 1) / s_value + Delta_d_value * (2 * M_value * phi_value + phi_value + 1) + (-Delta_p_value + E_value + f_value) * (M_value * phi_value + M_value + 2) <= 0] = np.nan
ax.plot_surface(Delta_d_value, Delta_p_value, SC_j_value, label="j", alpha=alpha)

# just to check calculations about Delta_d and Delta_p

Delta_d_dot_value = - (M_value + 2) / (M_value + 1) * delta_value * N_value / (phi_value * s_value)
Delta_p_dot_value = 1/(M_value + 1) * (-delta_value * N_value / (phi_value * s_value) + (M+1) * (f_value + E_value))

ax.scatter(Delta_d_dot_value, Delta_p_dot_value, 0)

ax.view_init(90, 0, 0)
ax.legend()

In [ ]:
(M_value * phi_value + M_value + 2) / (2 * M_value * phi_value + phi_value + 1)

In [ ]:
2 / (1 + phi_value)

In [ ]:
SC_j_value.max(axis=0).max()

In [ ]:
sols_i

In [ ]:
SC_expr.subs({N_d_var: sols_i.args[0].args[0], N_p_var: sols_i.args[0].args[1]}).simplify().diff(phi_var)

In [ ]:
sols_j

In [ ]:
to_simplify = SC_expr.subs({N_d_var: sols_j.args[0].args[0], N_p_var: sols_j.args[0].args[1]}).simplify().diff(phi_var).simplify()
to_simplify.args

In [ ]:
print(sym.latex(to_simplify.args[3].collect(phi_var))) # numerator to check that it is a polynome of varphi
to_simplify.args[3].collect(phi_var) # numerator

In [ ]:
to_simplify.args[3].collect(phi_var).collect(N_var)

In [ ]:
# simplify for N --> 0